# Ejercicio 8 — incorporación de 2025 y análisis completo

El notebook orquesta la descarga idempotente, procesamiento, validación, análisis de 2024–2026 y generación del informe Markdown. La primera ejecución puede tardar debido al tamaño de los 24 archivos de 2025.

## 1. Configuración

In [ ]:
from pathlib import Path
import subprocess
import sys
import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATABASE = ROOT / 'data' / 'processed' / 'ejercicio_6.duckdb'
DESCARGAR_2025 = True       # Cambie a False si los 24 archivos ya existen.
GENERAR_MARKDOWN = True
print(f'Proyecto: {ROOT}')

In [ ]:
def ejecutar_script(nombre, *argumentos):
    comando = [sys.executable, str(ROOT / 'scripts' / nombre), *argumentos]
    print('$', ' '.join(comando))
    resultado = subprocess.run(
        comando, cwd=ROOT, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )
    print(resultado.stdout)
    if resultado.returncode != 0:
        raise RuntimeError(f'{nombre} terminó con código {resultado.returncode}')
    return resultado

## 2. Descarga idempotente de 2025

El descargador revisa primero cada archivo local no vacío. Por ello, ejecutar esta celda nuevamente no descarga archivos ya existentes.

In [ ]:
if DESCARGAR_2025:
    ejecutar_script('download_data_ejercicio_8.py')
else:
    print('Descarga omitida por configuración.')
ejecutar_script('download_data_ejercicio_8.py', '--verify-only')

## 3. Prueba independiente de idempotencia

In [ ]:
ejecutar_script('verificar_idempotencia_descarga_8.py')

## 4. Procesamiento y materialización

Esta etapa crea los Parquet homologados de 2025, reconstruye `trips` con los seis conjuntos y vuelve a crear las vistas analíticas. Detenga Metabase antes de ejecutar si conserva una conexión de escritura.

In [ ]:
ejecutar_script('preparar_ejercicio_8.py')
ejecutar_script('validar_ejercicio_8.py')

## 5. Consultas analíticas

In [ ]:
con = duckdb.connect(str(DATABASE), read_only=True)

def consultar_sql(nombre):
    consulta = (ROOT / 'sql' / nombre).read_text(encoding='utf-8')
    return con.execute(consulta).df()

cobertura = consultar_sql('ejercicio_8_cobertura.sql')
anuales = consultar_sql('ejercicio_8_indicadores_anuales.sql')
comparables = consultar_sql('ejercicio_8_indicadores_comparables.sql')
mensuales = consultar_sql('ejercicio_8_evolucion_mensual.sql')
pagos = consultar_sql('ejercicio_8_formas_pago.sql')
display(Markdown('### Cobertura'))
display(cobertura)
display(Markdown('### Indicadores anuales observados'))
display(anuales)
display(Markdown('### Indicadores con ventana comparable'))
display(comparables)

## 6. Evolución mensual de viajes y ticket

In [ ]:
colores = {2024: '#4C78A8', 2025: '#F58518', 2026: '#54A24B'}
for metrica, titulo, eje_y in (
    ('viajes', 'Volumen mensual', 'Viajes'),
    ('ticket_mediano', 'Ticket mediano mensual', 'Importe'),
):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharex=True)
    for ax, taxi in zip(axes, ('yellow', 'green')):
        datos_taxi = mensuales[mensuales['taxi_type'] == taxi]
        for anio, datos in datos_taxi.groupby('data_year'):
            ax.plot(datos['mes'], datos[metrica], marker='o', markersize=3,
                    color=colores[int(anio)], label=str(anio))
        ax.set_title(taxi.capitalize())
        ax.set_xlabel('Mes')
        ax.set_ylabel(eje_y)
        ax.set_xticks(range(1, 13))
        ax.grid(alpha=.25)
        ax.legend(title='Año')
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()

## 7. Tres cambios o patrones cuantificados

In [ ]:
def porcentaje(inicial, final):
    return 100 * (final - inicial) / inicial if inicial else float('nan')

primero = int(comparables['data_year'].min())
ultimo = int(comparables['data_year'].max())
mes_limite = int(comparables['ultimo_mes_comparable'].min())
patrones = []
for columna, etiqueta in (
    ('viajes', 'volumen de viajes'),
    ('ticket_mediano', 'ticket mediano'),
    ('distancia_mediana', 'distancia mediana'),
):
    partes = []
    for taxi in ('yellow', 'green'):
        tabla = comparables[comparables['taxi_type'] == taxi].set_index('data_year')
        cambio = porcentaje(float(tabla.loc[primero, columna]), float(tabla.loc[ultimo, columna]))
        partes.append(f'{taxi}: {cambio:+.1f}%')
    patrones.append(f'**{etiqueta.capitalize()} ({primero}–{ultimo}):** ' + '; '.join(partes))
display(Markdown(
    f'Comparación enero–mes {mes_limite}:\n\n' +
    '\n'.join(f'{i}. {texto}' for i, texto in enumerate(patrones, 1))
))

## 8. Evolución de las formas de pago

In [ ]:
display(pagos)
tarjeta = pagos[pagos['forma_pago'] == 'Tarjeta']
fig, ax = plt.subplots(figsize=(9, 4.5))
for taxi, datos in tarjeta.groupby('taxi_type'):
    ax.plot(datos['data_year'], datos['porcentaje'], marker='o', label=taxi)
ax.set_xticks([2024, 2025, 2026])
ax.set_xlabel('Año')
ax.set_ylabel('% de viajes con tarjeta')
ax.set_title('Evolución del pago con tarjeta')
ax.grid(alpha=.25)
ax.legend(title='Taxi')
plt.show()

## 9. Generación del informe Markdown

In [ ]:
con.close()
if GENERAR_MARKDOWN:
    ejecutar_script('generar_reporte_ejercicio_8.py')
    reporte = ROOT / 'docs' / 'ejercicio_8_reporte.md'
    display(Markdown(f'Informe generado: `{reporte.relative_to(ROOT)}`'))
print('Ejercicio 8 ejecutado correctamente.')